## Téléchargement des trajets Webfleet

### Important
- Assurez-vous d’être connecté(e) au **bon compte Google (Gmail)**.
- Si vous **n’avez pas encore de clé API**, suivez d’abord la section **« Obtenir une clé API Webfleet »** ci-dessous.

---

### 1) Lancer le script (Google Colab)
- Ouvrez le notebook, puis exécutez les cellules.
- Colab va demander l’autorisation d’accès à **Google Drive** : **acceptez/autorisez**.
- Le Drive sera monté dans : `/content/drive`
- Cette autorisation est généralement requise **uniquement la première fois** (ou si vous changez de compte Google / révoquez l’accès).

---

### 2) Renseigner les informations demandées
Une fois le script lancé, le notebook vous demandera de saisir les paramètres suivants :

- **Date de début** (format `YYYY-MM-DD`)
- **Date de fin** (format `YYYY-MM-DD`)
- **Compte Webfleet** : `assistance-services`
- **Nom d’utilisateur Webfleet**
- **Mot de passe Webfleet** *(saisie masquée)*
- **Clé API Webfleet** *(saisie masquée)*
- **Nom du dossier de sortie** (ex. `WebfleetReports`)

---

### 3) Où trouver les fichiers générés
- Les fichiers sont enregistrés dans : **Google Drive → MyDrive → (votre dossier de sortie)**
- Deux fichiers Excel sont générés :
  - `webfleet_trips_daily_<DEBUT>_to_<FIN>.xlsx` *(1 onglet par jour)*
  - `webfleet_trips_all_<DEBUT>_to_<FIN>.xlsx` *(toutes les lignes combinées)*

---

## Obtenir une clé API Webfleet (si nécessaire)

### 1) Demande au support Webfleet
Faites une demande d’accès **WEBFLEET.connect / API key** via la page support ci-dessous :

- https://portals.webfleet.com/s/contactsupport?language=en_GB

Dans le formulaire :
- Utilisez **l’email associé au compte Webfleet** pour lequel vous souhaitez une clé API.
- **Subject** : `API key access`
- **Description** :

**Metez les informations ci-dessous** :

- **Application name** : HAMSautomations  
- **Integrator name** : Home Assistance Medical & Services  
- **Website** : https://homeassistance.ch/  
- **Contact person** : (votre nom)  
- **Address** : Rte Aloys-Fauquez 2, 1018 Lausanne  
- **Phone number** : 0800 94 94 94  
- **Email** : (votre email)  
- **Brief description of the application** :  
  Retrieves trip and vehicle usage data from Webfleet using the .connect API, processes it locally, and integrates the results into our existing internal tools. Please note that we are not integrating Webfleet into another established third-party platform, but rather using internal applications for flexible and automated data workflows, tailored to our specific needs.

---

### 2) Délai de réponse
Attendez la réponse du support Webfleet (généralement **1 à 2 jours ouvrables**).

---

### 3) Côté administrateur (obligatoire)
L’**administrateur principal** doit activer l’accès **WEBFLEET.connect / usage API** pour l’utilisateur concerné :

- **Paramètres utilisateurs** → sélectionner l’utilisateur → activer **WEBFLEET.connect / API** → **Enregistrer**

---

### 4) Telechargment success va etre notifier commes ci dessus


In [ ]:
import os
import re
import time
import csv
import requests
import pandas as pd
from io import StringIO
from tqdm.auto import tqdm
from getpass import getpass
from datetime import datetime, timedelta
import logging
import shutil

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# ============================================================
# CONFIG
# ============================================================
MOUNT_DRIVE = True
DRIVE_BASE_PATH = "/content/drive/MyDrive"

BASE_URL = "https://csv.webfleet.com/extern"
API_ACTION = "showTripReportExtern"
LANG = "en"
USE_ISO8601 = True
OUTPUTFORMAT = "csv"

# Webfleet docs say showTripReportExtern = 1 request / minute.
REQUEST_INTERVAL_SECS = 61

# Faster than daily, but still safe.
# If a weekly request fails/timeouts, the script automatically splits it smaller.
MAX_CHUNK_DAYS = 7

MAX_RETRIES = 5
TIMEOUT_SECS = 300

WRITE_XLSX_TOO = True
EXCEL_MAX_ROWS = 1_048_576

# ============================================================
# INPUTS
# ============================================================
START_DATE = input("Start date (YYYY-MM-DD): ").strip() or "2025-01-01"
END_DATE   = input("End date   (YYYY-MM-DD): ").strip() or "2025-12-31"

ACCOUNT   = input("Webfleet account: ").strip()
USERNAME  = input("Webfleet username: ").strip()
PASSWORD  = getpass("Webfleet password: ")
API_KEY   = getpass("Webfleet API key: ")

OUTPUT_FOLDER_NAME = input("Output folder name (e.g., 'WebfleetReports'): ").strip() or "WebfleetReports"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

    DRIVE_PATH = os.path.join(DRIVE_BASE_PATH, OUTPUT_FOLDER_NAME)
else:
    DRIVE_PATH = os.path.abspath(OUTPUT_FOLDER_NAME)

os.makedirs(DRIVE_PATH, exist_ok=True)

RUN_FOLDER = os.path.join(DRIVE_PATH, f"webfleet_year_download_{START_DATE}_to_{END_DATE}")
CACHE_FOLDER = os.path.join(RUN_FOLDER, "checkpoint_chunks")
os.makedirs(CACHE_FOLDER, exist_ok=True)

OUTPUT_CSV = os.path.join(RUN_FOLDER, f"webfleet_ALL_TRIPS_{START_DATE}_to_{END_DATE}.csv")
OUTPUT_XLSX = os.path.join(RUN_FOLDER, f"webfleet_ALL_TRIPS_{START_DATE}_to_{END_DATE}.xlsx")
MANIFEST_CSV = os.path.join(RUN_FOLDER, "download_manifest.csv")

print(f"\nOutput folder:\n{RUN_FOLDER}")
print(f"Checkpoint folder:\n{CACHE_FOLDER}\n")

# ============================================================
# SESSION
# ============================================================
session = requests.Session()
session.auth = (USERNAME, PASSWORD)
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/csv, text/plain, */*",
})

last_request_time = 0.0

# ============================================================
# HELPERS
# ============================================================
def to_date(s):
    return datetime.strptime(s, "%Y-%m-%d").date()

def date_str(d):
    return d.strftime("%Y-%m-%d")

def make_periods(start_str, end_str, max_days=7):
    start = to_date(start_str)
    end = to_date(end_str)

    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=max_days - 1), end)
        yield cur, chunk_end
        cur = chunk_end + timedelta(days=1)

def chunk_cache_path(start_d, end_d):
    return os.path.join(CACHE_FOLDER, f"trips_{date_str(start_d)}_to_{date_str(end_d)}.csv")

def chunk_empty_marker_path(start_d, end_d):
    return os.path.join(CACHE_FOLDER, f"EMPTY_{date_str(start_d)}_to_{date_str(end_d)}.txt")

def build_params(start_d, end_d):
    # Keep your original ISO-style date strings because your current script uses them.
    rf = f"{date_str(start_d)}T00:00:00"
    rt = f"{date_str(end_d)}T23:59:59"

    return {
        "account": ACCOUNT,
        "apikey": API_KEY,
        "action": API_ACTION,
        "outputformat": OUTPUTFORMAT,
        "lang": LANG,
        "useISO8601": str(USE_ISO8601).lower(),
        "rangefrom_string": rf,
        "rangeto_string": rt,
    }

def wait_for_rate_limit():
    global last_request_time

    elapsed = time.monotonic() - last_request_time
    if elapsed < REQUEST_INTERVAL_SECS:
        wait_s = REQUEST_INTERVAL_SECS - elapsed
        logging.info(f"Rate-limit pause: sleeping {wait_s:.1f}s")
        time.sleep(wait_s)

def looks_like_webfleet_error(text):
    if not text or not text.strip():
        return False

    first_line = text.strip().splitlines()[0].strip()

    # Webfleet body errors often look like 9000,... / 9006,... etc.
    # Avoid treating real CSV trip rows as errors.
    return bool(re.match(r"^9\d{3}\s*[,;]", first_line))

def parse_csv_robust(text, label):
    attempts = [
        dict(sep=None, engine="python"),
        dict(sep=";", engine="python", quoting=csv.QUOTE_MINIMAL),
        dict(sep=",", engine="python"),
        dict(sep=None, engine="python", on_bad_lines="skip"),
    ]

    last_exc = None

    for kwargs in attempts:
        try:
            df = pd.read_csv(StringIO(text), **kwargs)
            return df
        except Exception as e:
            last_exc = e

    raw_path = os.path.join(CACHE_FOLDER, f"RAW_PARSE_FAILED_{label}.csv")
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(text)

    raise RuntimeError(f"Could not parse CSV for {label}. Raw file saved to {raw_path}. Last error: {last_exc}")

def log_manifest(row):
    row = dict(row)
    row["logged_at"] = datetime.now().isoformat(timespec="seconds")

    df = pd.DataFrame([row])
    exists = os.path.exists(MANIFEST_CSV)
    df.to_csv(MANIFEST_CSV, mode="a", header=not exists, index=False)

def fetch_period_once(start_d, end_d):
    global last_request_time

    label = f"{date_str(start_d)}_to_{date_str(end_d)}"
    params = build_params(start_d, end_d)

    wait_for_rate_limit()

    logging.info(f"Requesting {label}")
    resp = session.get(BASE_URL, params=params, timeout=TIMEOUT_SECS)
    last_request_time = time.monotonic()

    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:500]}")

    text = resp.text

    if looks_like_webfleet_error(text):
        first_line = text.strip().splitlines()[0]

        if "document is empty" in first_line.lower():
            logging.info(f"{label}: Webfleet says document is empty.")
            return pd.DataFrame()

        raise RuntimeError(f"Webfleet error for {label}: {first_line}")

    if not text.strip():
        logging.info(f"{label}: Empty response.")
        return pd.DataFrame()

    df = parse_csv_robust(text, label)
    return df

def fetch_period_with_retries(start_d, end_d):
    label = f"{date_str(start_d)} to {date_str(end_d)}"
    last_exc = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = fetch_period_once(start_d, end_d)
            return df
        except Exception as e:
            last_exc = e
            logging.warning(f"{label}: attempt {attempt}/{MAX_RETRIES} failed: {e}")

            if attempt < MAX_RETRIES:
                time.sleep(min(10 * attempt, 60))

    raise RuntimeError(f"{label}: failed after {MAX_RETRIES} attempts. Last error: {last_exc}")

def save_chunk_atomic(df, start_d, end_d):
    path = chunk_cache_path(start_d, end_d)
    tmp_path = path + ".tmp"

    df = df.copy()
    df["__download_range_from"] = date_str(start_d)
    df["__download_range_to"] = date_str(end_d)

    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    os.replace(tmp_path, path)

    return path

def save_empty_marker(start_d, end_d):
    path = chunk_empty_marker_path(start_d, end_d)
    with open(path, "w", encoding="utf-8") as f:
        f.write("empty\n")
    return path

def fetch_period_safe(start_d, end_d):
    """
    Returns a list of checkpoint CSV paths.
    If a large chunk fails, split it into smaller chunks.
    """
    cached_csv = chunk_cache_path(start_d, end_d)
    cached_empty = chunk_empty_marker_path(start_d, end_d)

    if os.path.exists(cached_csv):
        logging.info(f"Already cached: {os.path.basename(cached_csv)}")
        return [cached_csv]

    if os.path.exists(cached_empty):
        logging.info(f"Already cached as empty: {os.path.basename(cached_empty)}")
        return []

    days_count = (end_d - start_d).days + 1
    label = f"{date_str(start_d)} to {date_str(end_d)}"

    try:
        df = fetch_period_with_retries(start_d, end_d)

        if df.empty:
            save_empty_marker(start_d, end_d)
            log_manifest({
                "from": date_str(start_d),
                "to": date_str(end_d),
                "status": "empty",
                "rows": 0,
                "file": "",
                "error": "",
            })
            return []

        path = save_chunk_atomic(df, start_d, end_d)

        log_manifest({
            "from": date_str(start_d),
            "to": date_str(end_d),
            "status": "ok",
            "rows": len(df),
            "file": path,
            "error": "",
        })

        logging.info(f"Saved checkpoint: {path} | rows={len(df)}")
        return [path]

    except Exception as e:
        logging.error(f"{label}: failed as one chunk: {e}")

        if days_count <= 1:
            log_manifest({
                "from": date_str(start_d),
                "to": date_str(end_d),
                "status": "failed",
                "rows": 0,
                "file": "",
                "error": str(e),
            })
            raise

        # Split the period into halves and retry.
        mid_d = start_d + timedelta(days=(days_count // 2) - 1)
        left_start, left_end = start_d, mid_d
        right_start, right_end = mid_d + timedelta(days=1), end_d

        logging.warning(f"Splitting {label} into {date_str(left_start)} to {date_str(left_end)} and {date_str(right_start)} to {date_str(right_end)}")

        files = []
        files.extend(fetch_period_safe(left_start, left_end))
        files.extend(fetch_period_safe(right_start, right_end))
        return files

def collect_union_columns(csv_files):
    cols = []

    for f in csv_files:
        header = pd.read_csv(f, nrows=0).columns.tolist()
        for c in header:
            if c not in cols:
                cols.append(c)

    return cols

def combine_checkpoint_csvs(csv_files, output_csv):
    if not csv_files:
        print("No trip rows found. Final file was not created.")
        return 0, []

    csv_files = sorted(csv_files)

    all_cols = collect_union_columns(csv_files)

    seen_tripids = set()
    use_tripid_dedupe = "tripid" in all_cols

    first_write = True
    total_rows = 0

    tmp_output = output_csv + ".tmp"

    if os.path.exists(tmp_output):
        os.remove(tmp_output)

    for f in tqdm(csv_files, desc="Combining checkpoint chunks"):
        for chunk in pd.read_csv(f, chunksize=50_000):
            chunk = chunk.reindex(columns=all_cols)

            if use_tripid_dedupe and "tripid" in chunk.columns:
                tripids = chunk["tripid"].astype(str)
                keep_mask = ~tripids.isin(seen_tripids)

                new_tripids = tripids[keep_mask].tolist()
                seen_tripids.update(new_tripids)

                chunk = chunk.loc[keep_mask]

            if chunk.empty:
                continue

            chunk.to_csv(
                tmp_output,
                mode="w" if first_write else "a",
                header=first_write,
                index=False,
                encoding="utf-8-sig",
            )

            first_write = False
            total_rows += len(chunk)

    os.replace(tmp_output, output_csv)
    return total_rows, all_cols

def write_xlsx_from_csv(input_csv, output_xlsx, total_rows):
    if total_rows == 0:
        return False

    if total_rows > EXCEL_MAX_ROWS - 1:
        print(f"⚠️ Too many rows for one Excel sheet: {total_rows:,}. CSV was created instead.")
        print(f"Excel one-sheet max data rows is {EXCEL_MAX_ROWS - 1:,} plus header.")
        return False

    from openpyxl import Workbook

    wb = Workbook(write_only=True)
    ws = wb.create_sheet("All Trips")

    first = True
    written_rows = 0

    for chunk in tqdm(pd.read_csv(input_csv, chunksize=50_000), desc="Writing one-sheet Excel"):
        if first:
            ws.append(list(chunk.columns))
            first = False

        for row in chunk.itertuples(index=False, name=None):
            ws.append(row)
            written_rows += 1

    wb.save(output_xlsx)
    return True

# ============================================================
# MAIN DOWNLOAD
# ============================================================
all_chunk_files = []

period_list = list(make_periods(START_DATE, END_DATE, max_days=MAX_CHUNK_DAYS))

print(f"Planned chunks: {len(period_list)}")
print(f"Chunk size: up to {MAX_CHUNK_DAYS} days")
print("Each successful chunk is saved immediately, so reruns can resume.\n")

failed = []

for start_d, end_d in tqdm(period_list, desc="Downloading periods"):
    try:
        files = fetch_period_safe(start_d, end_d)
        all_chunk_files.extend(files)
    except Exception as e:
        failed.append((date_str(start_d), date_str(end_d), str(e)))
        logging.error(f"FAILED permanently: {date_str(start_d)} to {date_str(end_d)} | {e}")

# Remove duplicate file paths while preserving order
seen_files = set()
unique_chunk_files = []
for f in all_chunk_files:
    if f not in seen_files:
        unique_chunk_files.append(f)
        seen_files.add(f)

# ============================================================
# FINAL COMBINE
# ============================================================
print("\nCombining all downloaded chunks into one master CSV...")

total_rows, final_cols = combine_checkpoint_csvs(unique_chunk_files, OUTPUT_CSV)

if total_rows > 0:
    print(f"\n✅ Saved master CSV:")
    print(OUTPUT_CSV)
    print(f"Rows: {total_rows:,}")

    if WRITE_XLSX_TOO:
        print("\nCreating one-sheet Excel file if it fits Excel limits...")
        made_xlsx = write_xlsx_from_csv(OUTPUT_CSV, OUTPUT_XLSX, total_rows)

        if made_xlsx:
            print(f"✅ Saved master Excel:")
            print(OUTPUT_XLSX)

if failed:
    print("\n⚠️ Some periods failed permanently:")
    for a, b, err in failed:
        print(f" - {a} to {b}: {err}")
    print("\nYou can rerun the same cell. Already cached chunks will be skipped.")
else:
    print("\n✅ All requested periods completed.")

print("\nDone.")

Start date (YYYY-MM-DD): 2026-04-01
End date   (YYYY-MM-DD): 2026-04-30
Webfleet account: assistance-services
Webfleet username: mkieffer@homeassistance.ch
Webfleet password: ··········
Webfleet API key: ··········
Output folder name (e.g., 'WebfleetReports'): april 2026
Mounted at /content/drive

Output folder:
/content/drive/MyDrive/april 2026/webfleet_year_download_2026-04-01_to_2026-04-30
Checkpoint folder:
/content/drive/MyDrive/april 2026/webfleet_year_download_2026-04-01_to_2026-04-30/checkpoint_chunks

Planned chunks: 5
Chunk size: up to 7 days
Each successful chunk is saved immediately, so reruns can resume.




Combining all downloaded chunks into one master CSV...


Combining checkpoint chunks:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Saved master CSV:
/content/drive/MyDrive/april 2026/webfleet_year_download_2026-04-01_to_2026-04-30/webfleet_ALL_TRIPS_2026-04-01_to_2026-04-30.csv
Rows: 6,068

Creating one-sheet Excel file if it fits Excel limits...


Writing one-sheet Excel: 0it [00:00, ?it/s]

✅ Saved master Excel:
/content/drive/MyDrive/april 2026/webfleet_year_download_2026-04-01_to_2026-04-30/webfleet_ALL_TRIPS_2026-04-01_to_2026-04-30.xlsx

✅ All requested periods completed.

Done.


In [ ]:
# ============================================================
# POST-DOWNLOAD AUDIT: CHECK IF ANY DATA LOOKS MISSING
# ============================================================

import os
import re
import glob
import pandas as pd
from datetime import datetime, timedelta
from collections import Counter

# This assumes these variables still exist from the download cell:
# START_DATE, END_DATE, RUN_FOLDER, CACHE_FOLDER, OUTPUT_CSV, MANIFEST_CSV

def to_date(s):
    return datetime.strptime(s, "%Y-%m-%d").date()

def date_str(d):
    return d.strftime("%Y-%m-%d")

def all_days_between(start_str, end_str):
    start = to_date(start_str)
    end = to_date(end_str)
    cur = start
    out = []
    while cur <= end:
        out.append(cur)
        cur += timedelta(days=1)
    return out

def count_csv_rows(path):
    # Counts data rows, excluding header
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        line_count = sum(1 for _ in f)
    return max(line_count - 1, 0)

def parse_period_from_filename(path):
    name = os.path.basename(path)

    m1 = re.match(r"trips_(\d{4}-\d{2}-\d{2})_to_(\d{4}-\d{2}-\d{2})\.csv$", name)
    if m1:
        return to_date(m1.group(1)), to_date(m1.group(2)), "data"

    m2 = re.match(r"EMPTY_(\d{4}-\d{2}-\d{2})_to_(\d{4}-\d{2}-\d{2})\.txt$", name)
    if m2:
        return to_date(m2.group(1)), to_date(m2.group(2)), "empty"

    return None

def days_in_period(start_d, end_d):
    cur = start_d
    while cur <= end_d:
        yield cur
        cur += timedelta(days=1)

# ------------------------------------------------------------
# 1) Find checkpoint files and empty markers
# ------------------------------------------------------------
chunk_files = sorted(glob.glob(os.path.join(CACHE_FOLDER, "trips_*.csv")))
empty_markers = sorted(glob.glob(os.path.join(CACHE_FOLDER, "EMPTY_*.txt")))

print("========== WEBFLEET DOWNLOAD AUDIT ==========")
print(f"Requested range: {START_DATE} to {END_DATE}")
print(f"Checkpoint CSV chunks found: {len(chunk_files)}")
print(f"Empty markers found: {len(empty_markers)}")
print(f"Final output CSV exists: {os.path.exists(OUTPUT_CSV)}")
print()

# ------------------------------------------------------------
# 2) Check date coverage
# ------------------------------------------------------------
requested_days = set(all_days_between(START_DATE, END_DATE))
covered_days = set()
data_days = set()
empty_days = set()

period_records = []

for path in chunk_files + empty_markers:
    parsed = parse_period_from_filename(path)
    if parsed is None:
        continue

    start_d, end_d, kind = parsed

    for d in days_in_period(start_d, end_d):
        covered_days.add(d)
        if kind == "data":
            data_days.add(d)
        elif kind == "empty":
            empty_days.add(d)

    period_records.append({
        "file": os.path.basename(path),
        "from": date_str(start_d),
        "to": date_str(end_d),
        "kind": kind,
    })

missing_coverage = sorted(requested_days - covered_days)
extra_coverage = sorted(covered_days - requested_days)

if not missing_coverage:
    print("✅ Date coverage check: every requested day is covered by a checkpoint or empty marker.")
else:
    print("❌ Date coverage check FAILED. These requested days have no checkpoint and no empty marker:")
    for d in missing_coverage:
        print(" -", date_str(d))

if extra_coverage:
    print("\n⚠️ Extra coverage found outside requested range:")
    for d in extra_coverage:
        print(" -", date_str(d))

print()

# ------------------------------------------------------------
# 3) Check manifest for failed periods
# ------------------------------------------------------------
manifest_failed = pd.DataFrame()

if os.path.exists(MANIFEST_CSV):
    manifest = pd.read_csv(MANIFEST_CSV)

    if "status" in manifest.columns:
        manifest_failed = manifest[manifest["status"].astype(str).str.lower().eq("failed")]

    if manifest_failed.empty:
        print("✅ Manifest check: no failed periods found.")
    else:
        print("❌ Manifest check FAILED. Failed periods found:")
        display(manifest_failed)
else:
    print("⚠️ Manifest file not found. Cannot check logged failed periods.")

print()

# ------------------------------------------------------------
# 4) Compare raw checkpoint row count vs final output row count
# ------------------------------------------------------------
raw_checkpoint_rows = sum(count_csv_rows(f) for f in chunk_files)
final_rows = count_csv_rows(OUTPUT_CSV) if os.path.exists(OUTPUT_CSV) else 0

print("Row count check:")
print(f" - Raw checkpoint rows: {raw_checkpoint_rows:,}")
print(f" - Final output rows:   {final_rows:,}")

# ------------------------------------------------------------
# 5) Trip ID audit, if tripid exists
# ------------------------------------------------------------
tripid_exists = False
raw_tripids = []
duplicate_tripids = Counter()

for f in chunk_files:
    try:
        header = pd.read_csv(f, nrows=0).columns.tolist()
    except Exception:
        continue

    if "tripid" in header:
        tripid_exists = True

        for chunk in pd.read_csv(f, usecols=["tripid"], chunksize=100_000):
            ids = chunk["tripid"].dropna().astype(str).tolist()
            raw_tripids.extend(ids)

if tripid_exists:
    raw_counter = Counter(raw_tripids)
    duplicate_ids = {k: v for k, v in raw_counter.items() if v > 1}
    unique_raw_tripids = set(raw_counter.keys())

    final_tripids = set()

    if os.path.exists(OUTPUT_CSV):
        final_header = pd.read_csv(OUTPUT_CSV, nrows=0).columns.tolist()

        if "tripid" in final_header:
            for chunk in pd.read_csv(OUTPUT_CSV, usecols=["tripid"], chunksize=100_000):
                final_tripids.update(chunk["tripid"].dropna().astype(str).tolist())

    missing_tripids_in_final = sorted(unique_raw_tripids - final_tripids)

    expected_final_rows_if_deduped = len(unique_raw_tripids)

    print()
    print("Trip ID check:")
    print(f" - Raw tripid rows:             {len(raw_tripids):,}")
    print(f" - Unique raw tripids:          {len(unique_raw_tripids):,}")
    print(f" - Duplicate tripids in chunks: {len(duplicate_ids):,}")
    print(f" - Final tripids:               {len(final_tripids):,}")

    if not missing_tripids_in_final:
        print("✅ Trip ID check: every unique checkpoint tripid exists in the final output.")
    else:
        print("❌ Trip ID check FAILED. Some trip IDs from checkpoints are missing in the final file.")
        print("Showing first 50 missing trip IDs:")
        print(missing_tripids_in_final[:50])

    if final_rows == expected_final_rows_if_deduped:
        print("✅ Final row count matches unique tripid count.")
    else:
        print("⚠️ Final row count does not exactly match unique tripid count.")
        print("   This may be okay if some rows have blank tripid, but you should inspect it.")

    if duplicate_ids:
        duplicate_report_path = os.path.join(RUN_FOLDER, "duplicate_tripids_found_in_checkpoints.csv")
        pd.DataFrame(
            [{"tripid": k, "count": v} for k, v in duplicate_ids.items()]
        ).to_csv(duplicate_report_path, index=False, encoding="utf-8-sig")
        print(f"⚠️ Duplicate tripid report saved to: {duplicate_report_path}")

else:
    print()
    print("⚠️ No 'tripid' column found in checkpoint files.")
    print("   Cannot do the strongest duplicate/missing trip ID audit.")
    if raw_checkpoint_rows == final_rows:
        print("✅ Basic row count check passed: checkpoint rows equal final rows.")
    else:
        print("❌ Basic row count check failed: checkpoint rows do not equal final rows.")

print()

# ------------------------------------------------------------
# 6) Show zero-data days / empty days
# ------------------------------------------------------------
empty_days_sorted = sorted(empty_days & requested_days)

if empty_days_sorted:
    print("Days marked as officially empty:")
    for d in empty_days_sorted:
        print(" -", date_str(d))
else:
    print("✅ No days were marked empty.")

print()

# ------------------------------------------------------------
# 7) Final verdict
# ------------------------------------------------------------
hard_fail = False

if missing_coverage:
    hard_fail = True

if not manifest_failed.empty:
    hard_fail = True

if tripid_exists:
    if missing_tripids_in_final:
        hard_fail = True
else:
    if raw_checkpoint_rows != final_rows:
        hard_fail = True

print("========== FINAL AUDIT RESULT ==========")

if hard_fail:
    print("❌ AUDIT FAILED: There is evidence that data may be missing or the final file is incomplete.")
    print("Recommended action: rerun the download cell. Cached successful chunks will be skipped.")
else:
    print("✅ AUDIT PASSED: No missing coverage, no failed periods, and checkpoint data matches the final output.")
    print("This is a strong confirmation that the final file contains all successfully downloaded Webfleet data.")

========== WEBFLEET DOWNLOAD AUDIT ==========
Requested range: 2026-04-01 to 2026-04-30
Checkpoint CSV chunks found: 5
Empty markers found: 0
Final output CSV exists: True

✅ Date coverage check: every requested day is covered by a checkpoint or empty marker.

✅ Manifest check: no failed periods found.

Row count check:
 - Raw checkpoint rows: 6,068
 - Final output rows:   6,068

Trip ID check:
 - Raw tripid rows:             6,068
 - Unique raw tripids:          6,068
 - Duplicate tripids in chunks: 0
 - Final tripids:               6,068
✅ Trip ID check: every unique checkpoint tripid exists in the final output.
✅ Final row count matches unique tripid count.

✅ No days were marked empty.

========== FINAL AUDIT RESULT ==========
✅ AUDIT PASSED: No missing coverage, no failed periods, and checkpoint data matches the final output.
This is a strong confirmation that the final file contains all successfully downloaded Webfleet data.


In [ ]:
import os
import time
import csv
import requests
import pandas as pd
from io import StringIO
from tqdm.auto import tqdm
from getpass import getpass
from datetime import datetime, timedelta
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ------------------------------
# CONFIG (edit if you like) with daily file
# ------------------------------
MOUNT_DRIVE = True
DRIVE_BASE_PATH = "/content/drive/MyDrive"
DELAY_BETWEEN_REQUESTS = 60
MAX_RETRIES = 10
TIMEOUT_SECS = 300

# API config
BASE_URL = "https://csv.webfleet.com/extern"
API_ACTION = "showTripReportExtern"
LANG = "en"
USE_ISO8601 = True
OUTPUTFORMAT = "csv"

# ------------------------------
# Inputs (dates + credentials)
# ------------------------------
START_DATE = input("Start date (YYYY-MM-DD): ").strip() or "2025-07-01"
END_DATE   = input("End date   (YYYY-MM-DD): ").strip() or "2025-07-07"

ACCOUNT   = input("Webfleet account: ").strip()
USERNAME  = input("Webfleet username: ").strip()
PASSWORD  = getpass("Webfleet password: ")
API_KEY   = getpass("Webfleet API key: ")

OUTPUT_FOLDER_NAME = input("Output folder name (e.g., 'WebfleetReports'): ").strip() or "WebfleetReports"

# Output filenames
OUTPUT_XLSX_DAILY = f"webfleet_trips_daily_{START_DATE}_to_{END_DATE}.xlsx"
OUTPUT_XLSX_ALL   = f"webfleet_trips_all_{START_DATE}_to_{END_DATE}.xlsx"

# Optional: mount Drive
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_PATH = os.path.join(DRIVE_BASE_PATH, OUTPUT_FOLDER_NAME)
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f"Output will be saved to: {DRIVE_PATH}")

    OUTPUT_XLSX_DAILY = os.path.join(DRIVE_PATH, OUTPUT_XLSX_DAILY)
    OUTPUT_XLSX_ALL   = os.path.join(DRIVE_PATH, OUTPUT_XLSX_ALL)

print("\nConfig OK. Starting download...\n")

# ------------------------------
# Session with Basic Auth
# ------------------------------
session = requests.Session()
session.auth = (USERNAME, PASSWORD)
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "text/csv, text/plain, */*",
})

# ------------------------------
# Helpers
# ------------------------------
def daterange(start_date_str, end_date_str):
    start = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    end   = datetime.strptime(end_date_str,   "%Y-%m-%d").date()
    cur = start
    while cur <= end:
        yield cur
        cur += timedelta(days=1)

def build_params(day_str):
    rf = f"{day_str}T00:00:00"
    rt = f"{day_str}T23:59:59"
    return {
        "account": ACCOUNT,
        "apikey": API_KEY,
        "action": API_ACTION,
        "outputformat": OUTPUTFORMAT,
        "lang": LANG,
        "useISO8601": str(USE_ISO8601).lower(),
        "rangefrom_string": rf,
        "rangeto_string":   rt,
    }

def looks_like_webfleet_error(text):
    """
    Webfleet errors often look like: "9xxx,<message>"
    We'll treat a first token of digits + comma as likely error.
    """
    if not text:
        return False
    first_line = text.strip().splitlines()[0]
    return first_line and first_line.split(",")[0].isdigit()

def parse_csv_robust(text, day_str):
    """
    Try multiple parsing strategies to survive inconsistent delimiters/quotes.
    """
    attempts = [
        dict(args=dict(sep=None, engine="python"), kwargs={}),
        dict(args=dict(sep=";",  engine="python", quoting=csv.QUOTE_MINIMAL), kwargs={}),
        dict(args=dict(sep=None, engine="python", on_bad_lines="skip"), kwargs={}),
        dict(args=dict(sep=",", engine="python"), kwargs={}),
    ]

    last_exc = None
    for a in attempts:
        try:
            return pd.read_csv(StringIO(text), **a["args"], **a["kwargs"])
        except Exception as e:
            last_exc = e

    # If nothing worked, dump raw for inspection
    fname = f"raw_{day_str}.csv"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(text)
    raise RuntimeError(f"Failed to parse CSV for {day_str}. Saved {fname}. Last error: {last_exc}")

def fetch_csv_for_day(day_str, max_retries=MAX_RETRIES, timeout=TIMEOUT_SECS):
    params = build_params(day_str)
    last_exc = None

    for attempt in range(1, max_retries + 1):
        try:
            resp = session.get(BASE_URL, params=params, timeout=timeout)

            if resp.status_code != 200:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")

            text = resp.text

            # Detect Webfleet body-level error
            if looks_like_webfleet_error(text):
                first_line = text.strip().splitlines()[0]

                if "document is empty" in first_line.lower():
                    logging.info(f"[{day_str}] No data returned (document is empty).")
                    return pd.DataFrame()

                raise RuntimeError(f"{day_str}: Webfleet error -> {first_line}")

            if not text.strip():
                logging.info(f"[{day_str}] No data returned (empty CSV).")
                return pd.DataFrame()

            df = parse_csv_robust(text, day_str)
            logging.info(f"[{day_str}] Successfully fetched and parsed data. Rows: {len(df)}.")
            return df

        except Exception as e:
            last_exc = e
            sleep_s = 3 * attempt
            logging.warning(f"[{day_str}] Attempt {attempt}/{max_retries}: Error fetching data: {e}. Retrying in {sleep_s}s...")
            time.sleep(sleep_s)

    logging.error(f"[{day_str}] Failed to fetch data after {max_retries} attempts. Last error: {last_exc}")
    raise last_exc

# ------------------------------
# Main: fetch days
# ------------------------------
all_results = {}   # {sheet_name: DataFrame}
all_combined = []
failed_days = []

dates = [d for d in daterange(START_DATE, END_DATE)]

for i, d in enumerate(tqdm(dates, desc="Downloading days"), start=1):
    day_str = d.strftime("%Y-%m-%d")
    try:
        df = fetch_csv_for_day(day_str)
        all_results[day_str] = df
        if not df.empty:
            df = df.copy()
            df["__download_day"] = day_str
            all_combined.append(df)
    except Exception as e:
        failed_days.append((day_str, str(e)))
        logging.error(f"!! Skipping {day_str} due to error: {e}")

    # Polite pause between requests, but not after the last day
    if i < len(dates):
        time.sleep(DELAY_BETWEEN_REQUESTS)

if not all_results:
    raise SystemExit("No data downloaded; nothing to write.")

# ------------------------------
# Write daily workbook (one sheet per day)
# ------------------------------
with pd.ExcelWriter(OUTPUT_XLSX_DAILY, engine="openpyxl") as writer:
    for sheet_name, df in all_results.items():
        safe_sheet = sheet_name[:31]
        df.to_excel(writer, sheet_name=safe_sheet, index=False)

print(f"\n✅ Saved daily sheets -> {OUTPUT_XLSX_DAILY}")

# ------------------------------
# Write all-in-one workbook (single sheet)
# ------------------------------
if all_combined:
    combined_df = pd.concat(all_combined, ignore_index=True)
    combined_df.to_excel(OUTPUT_XLSX_ALL, index=False)
    print(f"✅ Saved all records (single sheet) -> {OUTPUT_XLSX_ALL}\n")
else:
    print("ℹ️ No rows found across the range; skipped all-in-one file.\n")

# ------------------------------
# Print summary of failed days
# ------------------------------
if failed_days:
    print("⚠️ Failed days:")
    for day_str, err in failed_days:
        print(f" - {day_str}: {err}")
else:
    print("✅ All days downloaded successfully.")

print("Done.")

Start date (YYYY-MM-DD): 2026-04-01
End date   (YYYY-MM-DD): 2026-04-30
Webfleet account: assistance-services
Webfleet username: mkieffer@homeassistance.ch
Webfleet password: ··········
Webfleet API key: ··········
Output folder name (e.g., 'WebfleetReports'): april 2026 v2
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output will be saved to: /content/drive/MyDrive/april 2026 v2

Config OK. Starting download...



KeyboardInterrupt: 